# 09 -- Thread Export for Bot Training

This notebook extracts conversation chains from the r/litigi comment tree
and exports them as structured training data for language model fine-tuning.

Reddit threads have a natural tree structure: each submission is the root,
and each comment is a child of either the submission or another comment.
Every root-to-leaf path in the tree is a conversation chain that can serve
as a training sequence.

Two export formats are produced:

1. **Full chains** (`output/litigi_threads.jsonl`): one JSON object per
   root-to-leaf path, containing the thread ID and the ordered list of
   comment bodies and authors.

2. **Prompt/response pairs** (`output/litigi_pairs.jsonl`): each adjacent
   (comment, reply) pair within a chain becomes one training sample.
   Deleted or removed comments are filtered out automatically.

## Prerequisites

Run `02_data_loading.ipynb` first to produce `data/litigi_comments.parquet`.

In [ ]:
from pathlib import Path

from subreddit_lens import create_nx_graph, load_comments
from subreddit_lens.export import (
    chains_to_prompt_pairs,
    export_chains_to_jsonl,
    export_prompt_pairs_to_jsonl,
    extract_thread_chains,
)

# Locate the repository root so paths work regardless of the working directory.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
DATA_FILE = DATA_DIR / "litigi_comments.parquet"
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. Load Data

In [ ]:
df = load_comments(DATA_FILE)
print(f"Loaded {len(df):,} comments from {len(df['author'].unique()):,} authors")
print(f"Date range: {df['created_dt'].min()} to {df['created_dt'].max()}")
df[["id", "author", "body", "parent_id", "link_id", "created_dt"]].head()

## 2. Build the Comment Thread Graph

The graph has one node per comment (by ID) and one node per submission.
Edges point from parent to child, following the natural reading order of
the conversation.

In [ ]:
G = create_nx_graph(df)
print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

# Count submission roots (nodes with no incoming edges)
root_nodes = [n for n in G.nodes() if G.in_degree(n) == 0]
print(f"Submission roots: {len(root_nodes):,}")

## 3. Extract Conversation Chains

Every root-to-leaf path in the thread forest is extracted as a conversation
chain. The `min_length` parameter controls the minimum number of comments
required in a chain. Chains of length 1 (single top-level comments with no
replies) are not useful for training.

In [ ]:
chains = extract_thread_chains(G, df, min_length=2)
print(f"Extracted {len(chains):,} conversation chains")

chain_lengths = [len(c["chain"]) for c in chains]
import numpy as np
print(f"Chain length -- min: {min(chain_lengths)}, max: {max(chain_lengths)}, "
      f"mean: {np.mean(chain_lengths):.1f}, median: {np.median(chain_lengths):.1f}")

In [ ]:
# Inspect a sample chain
sample = chains[0]
print(f"Thread ID: {sample['thread_id']}")
print(f"Chain length: {len(sample['chain'])} comments\n")
for i, comment in enumerate(sample["chain"]):
    print(f"[{i}] {comment['author']}: {comment['body'][:120]}")

## 4. Export Full Chains to JSONL

Each line of the output file is a JSON object containing the thread ID and
the ordered list of comments in that chain.

In [ ]:
chains_path = OUTPUT_DIR / "litigi_threads.jsonl"
export_chains_to_jsonl(chains, chains_path)

## 5. Convert to Prompt/Response Pairs

Each adjacent (comment, reply) pair within a chain becomes one training
sample. Pairs involving deleted or removed comments are excluded.

The `system_prompt` argument sets a system-level instruction that will
appear in every training sample. Leave it empty if you do not use a
system prompt in your training setup.

In [ ]:
SYSTEM_PROMPT = ""  # Set a system prompt here if desired, e.g. "Sei un utente di r/litigi."

pairs = chains_to_prompt_pairs(chains, system_prompt=SYSTEM_PROMPT, skip_removed=True)
print(f"Generated {len(pairs):,} prompt/response pairs")

In [ ]:
# Inspect a few pairs
for pair in pairs[:3]:
    print("USER:", pair["user"][:100])
    print("ASSISTANT:", pair["assistant"][:100])
    print()

In [ ]:
pairs_path = OUTPUT_DIR / "litigi_pairs.jsonl"
export_prompt_pairs_to_jsonl(pairs, pairs_path)

## 6. Summary

The exported files are in `output/`:

- `litigi_threads.jsonl` -- full conversation chains for inspection or
  custom processing.
- `litigi_pairs.jsonl` -- prompt/response pairs ready for supervised
  fine-tuning.

Each line of `litigi_pairs.jsonl` follows the format:
```json
{"system": "", "user": "<comment body>", "assistant": "<reply body>"}
```